# TTM T3 — Colab/Google Drive checkpoint/resume, 100% synthetic

**Không tải TinyStories, không huấn luyện khoa học.** Notebook ghi checkpoint PyTorch đầy đủ lên Google Drive: model/optimizer/scheduler/RNG/cursor. So sánh 20 bước liên tục với 10 bước + lưu + đọc lại từ Drive + 10 bước, dùng đúng mã P02 đã khóa. **T4 khoa học chưa được cấp phép.**

Chạy từng ô một. Notebook chỉ dùng GPU nếu Colab cấp CUDA và sẽ từ chối chạy âm thầm trên CPU. Không chạy lại cùng run ID sau lỗi. Cuối cùng phải flush/unmount Drive và kiểm chứng file trên Drive độc lập.


In [ ]:
import os, sys, base64, hashlib, pathlib, subprocess, json
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
ROOT=pathlib.Path('/content/ttm_t3_frozen')
ROOT.mkdir(parents=True,exist_ok=True)
raw=base64.b64decode('IiIiVFRNIFAwMiBkZWNvZGVyLW9ubHkgVHJhbnNmb3JtZXIgY29yZS4gQ1BVLWNvcnJlY3RuZXNzIGNhbmRpZGF0ZSwgbm90IFAwMyB0cmFpbmluZy4KCkJQRSBpcyBmcm96ZW4gYXQgMjA0OCB0b2tlbiBJRHM7IHRoaXMgbW9kdWxlIGRvZXMgbm90IHJlYWQgY29ycHVzIG9yIGhlbGQtb3V0IFRFU1QuClRoZSBMYXllck5vcm0gdmFyaWFudCBpbnRlbnRpb25hbGx5IGhhcyB0aGUgc2FtZSBwYXJhbWV0ZXIgbmFtZXMgYW5kIG1hdGggYXMgUDAwLgoiIiIKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmltcG9ydCBtYXRoCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KZnJvbSB0b3JjaC5ubiBpbXBvcnQgZnVuY3Rpb25hbCBhcyBGCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgUDAyQ29uZmlnOgogICAgdm9jYWJfc2l6ZTogaW50ID0gMjA0OAogICAgbWF4X3NlcV9sZW46IGludCA9IDY0CiAgICBuX2VtYmQ6IGludCA9IDY0CiAgICBuX2xheWVyOiBpbnQgPSAyCiAgICBuX2hlYWQ6IGludCA9IDQKICAgIG5vcm1fdHlwZTogc3RyID0gImxheWVybm9ybSIKCiAgICBkZWYgX19wb3N0X2luaXRfXyhzZWxmKToKICAgICAgICBpZiBub3QgYWxsKGlzaW5zdGFuY2UodiwgaW50KSBhbmQgbm90IGlzaW5zdGFuY2UodiwgYm9vbCkgYW5kIHYgPiAwCiAgICAgICAgICAgICAgICAgICBmb3IgdiBpbiAoc2VsZi52b2NhYl9zaXplLCBzZWxmLm1heF9zZXFfbGVuLCBzZWxmLm5fZW1iZCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzZWxmLm5fbGF5ZXIsIHNlbGYubl9oZWFkKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImFsbCBtb2RlbCBkaW1lbnNpb25zIG11c3QgYmUgcG9zaXRpdmUgaW50ZWdlcnMiKQogICAgICAgIGlmIHNlbGYubl9lbWJkICUgc2VsZi5uX2hlYWQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5fZW1iZCBtdXN0IGJlIGRpdmlzaWJsZSBieSBuX2hlYWQiKQogICAgICAgIGlmIHNlbGYubm9ybV90eXBlIG5vdCBpbiAoImxheWVybm9ybSIsICJybXNub3JtIik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5vcm1fdHlwZSBtdXN0IGJlIGxheWVybm9ybSBvciBybXNub3JtIikKCgpjbGFzcyBSTVNOb3JtKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgZGltOiBpbnQsIGVwczogZmxvYXQgPSAxZS01KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmVwcyA9IGVwcwogICAgICAgIHNlbGYud2VpZ2h0ID0gbm4uUGFyYW1ldGVyKHRvcmNoLm9uZXMoZGltKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBzY2FsZSA9IHRvcmNoLnJzcXJ0KHguc3F1YXJlKCkubWVhbihkaW09LTEsIGtlZXBkaW09VHJ1ZSkgKyBzZWxmLmVwcykKICAgICAgICByZXR1cm4geCAqIHNjYWxlICogc2VsZi53ZWlnaHQKCgpkZWYgX25vcm0oY2ZnOiBQMDJDb25maWcpIC0+IG5uLk1vZHVsZToKICAgIGlmIGNmZy5ub3JtX3R5cGUgPT0gImxheWVybm9ybSI6CiAgICAgICAgcmV0dXJuIG5uLkxheWVyTm9ybShjZmcubl9lbWJkKQogICAgcmV0dXJuIFJNU05vcm0oY2ZnLm5fZW1iZCkKCgpjbGFzcyBDYXVzYWxTZWxmQXR0ZW50aW9uKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgY2ZnOiBQMDJDb25maWcpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuaGVhZHMgPSBjZmcubl9oZWFkCiAgICAgICAgc2VsZi5oZWFkX2RpbSA9IGNmZy5uX2VtYmQgLy8gY2ZnLm5faGVhZAogICAgICAgIHNlbGYucWt2ID0gbm4uTGluZWFyKGNmZy5uX2VtYmQsIDMgKiBjZmcubl9lbWJkKQogICAgICAgIHNlbGYucHJvaiA9IG5uLkxpbmVhcihjZmcubl9lbWJkLCBjZmcubl9lbWJkKQoKICAgIGRlZiBfc3BsaXQoc2VsZiwgdDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgYmF0Y2gsIHRpbWUsIGNoYW5uZWxzID0gdC5zaGFwZQogICAgICAgIHJldHVybiB0LnJlc2hhcGUoYmF0Y2gsIHRpbWUsIHNlbGYuaGVhZHMsIHNlbGYuaGVhZF9kaW0pLnRyYW5zcG9zZSgxLCAyKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIGJhdGNoLCB0aW1lLCBjaGFubmVscyA9IHguc2hhcGUKICAgICAgICBxLCBrLCB2ID0gW3NlbGYuX3NwbGl0KGEpIGZvciBhIGluIHNlbGYucWt2KHgpLmNodW5rKDMsIGRpbT0tMSldCiAgICAgICAgc2NvcmVzID0gcSBAIGsudHJhbnNwb3NlKC0yLCAtMSkgLyBtYXRoLnNxcnQoc2VsZi5oZWFkX2RpbSkKICAgICAgICBtYXNrID0gdG9yY2gub25lcyh0aW1lLCB0aW1lLCBkdHlwZT10b3JjaC5ib29sLCBkZXZpY2U9eC5kZXZpY2UpLnRyaWwoKQogICAgICAgIHdlaWdodHMgPSBzY29yZXMubWFza2VkX2ZpbGwofm1hc2ssIC10b3JjaC5pbmYpLnNvZnRtYXgoZGltPS0xKQogICAgICAgIHkgPSB3ZWlnaHRzIEAgdgogICAgICAgIHkgPSB5LnRyYW5zcG9zZSgxLCAyKS5jb250aWd1b3VzKCkucmVzaGFwZShiYXRjaCwgdGltZSwgY2hhbm5lbHMpCiAgICAgICAgcmV0dXJuIHNlbGYucHJvaih5KQoKCmNsYXNzIEJsb2NrKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgY2ZnOiBQMDJDb25maWcpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubG4xID0gX25vcm0oY2ZnKQogICAgICAgIHNlbGYuYXR0biA9IENhdXNhbFNlbGZBdHRlbnRpb24oY2ZnKQogICAgICAgIHNlbGYubG4yID0gX25vcm0oY2ZnKQogICAgICAgIHNlbGYubWxwID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKGNmZy5uX2VtYmQsIDQgKiBjZmcubl9lbWJkKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5MaW5lYXIoNCAqIGNmZy5uX2VtYmQsIGNmZy5uX2VtYmQpLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICB4ID0geCArIHNlbGYuYXR0bihzZWxmLmxuMSh4KSkKICAgICAgICByZXR1cm4geCArIHNlbGYubWxwKHNlbGYubG4yKHgpKQoKCmNsYXNzIEdQVChubi5Nb2R1bGUpOgogICAgIiIiUDAyIGNvcmUgd2l0aCBQMDAtY29tcGF0aWJsZSBMYXllck5vcm0gcGFyYW1ldGVyIG5hbWVzL3N0YXRlX2RpY3QuIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2ZnOiBQMDJDb25maWcpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuY2ZnID0gY2ZnCiAgICAgICAgc2VsZi50b2sgPSBubi5FbWJlZGRpbmcoY2ZnLnZvY2FiX3NpemUsIGNmZy5uX2VtYmQpCiAgICAgICAgc2VsZi5wb3MgPSBubi5FbWJlZGRpbmcoY2ZnLm1heF9zZXFfbGVuLCBjZmcubl9lbWJkKQogICAgICAgIHNlbGYuYmxvY2tzID0gbm4uTW9kdWxlTGlzdChbQmxvY2soY2ZnKSBmb3IgXyBpbiByYW5nZShjZmcubl9sYXllcildKQogICAgICAgIHNlbGYuZmluYWxfbG4gPSBfbm9ybShjZmcpCiAgICAgICAgc2VsZi5oZWFkID0gbm4uTGluZWFyKGNmZy5uX2VtYmQsIGNmZy52b2NhYl9zaXplLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYuaGVhZC53ZWlnaHQgPSBzZWxmLnRvay53ZWlnaHQKICAgICAgICBzZWxmLmFwcGx5KHNlbGYuX2luaXRfd2VpZ2h0cykKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX2luaXRfd2VpZ2h0cyhtb2R1bGU6IG5uLk1vZHVsZSkgLT4gTm9uZToKICAgICAgICBpZiBpc2luc3RhbmNlKG1vZHVsZSwgKG5uLkxpbmVhciwgbm4uRW1iZWRkaW5nKSk6CiAgICAgICAgICAgIG5uLmluaXQubm9ybWFsXyhtb2R1bGUud2VpZ2h0LCBtZWFuPTAuMCwgc3RkPTAuMDIpCiAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UobW9kdWxlLCBubi5MaW5lYXIpIGFuZCBtb2R1bGUuYmlhcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIG5uLmluaXQuemVyb3NfKG1vZHVsZS5iaWFzKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHRva2VuczogdG9yY2guVGVuc29yLCB0YXJnZXRzOiB0b3JjaC5UZW5zb3IgfCBOb25lID0gTm9uZSk6CiAgICAgICAgaWYgdG9rZW5zLmR0eXBlICE9IHRvcmNoLmxvbmcgb3IgdG9rZW5zLm5kaW0gIT0gMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidG9rZW5zIG11c3QgYmUgTG9uZ1RlbnNvciBbYmF0Y2gsIHRpbWVdIikKICAgICAgICBpZiB0b2tlbnMuc2hhcGVbMF0gPCAxIG9yIG5vdCAoMSA8PSB0b2tlbnMuc2hhcGVbMV0gPD0gc2VsZi5jZmcubWF4X3NlcV9sZW4pOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0b2tlbnMgbXVzdCBoYXZlIG5vbmVtcHR5IGJhdGNoL3RpbWUgd2l0aGluIG1heF9zZXFfbGVuIikKICAgICAgICBpZiB0b3JjaC5hbnkodG9rZW5zIDwgMCkuaXRlbSgpIG9yIHRvcmNoLmFueSh0b2tlbnMgPj0gc2VsZi5jZmcudm9jYWJfc2l6ZSkuaXRlbSgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0b2tlbiBJRCBvdXRzaWRlIFAwMiB2b2NhYnVsYXJ5IikKICAgICAgICBpZiB0YXJnZXRzIGlzIG5vdCBOb25lIGFuZCAodGFyZ2V0cy5kdHlwZSAhPSB0b3JjaC5sb25nIG9yIHRhcmdldHMuc2hhcGUgIT0gdG9rZW5zLnNoYXBlKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidGFyZ2V0cyBtdXN0IG1hdGNoIHRva2VucyBzaGFwZSBhbmQgZHR5cGUiKQogICAgICAgIHRpbWUgPSB0b2tlbnMuc2hhcGVbMV0KICAgICAgICBwb3NpdGlvbnMgPSB0b3JjaC5hcmFuZ2UodGltZSwgZGV2aWNlPXRva2Vucy5kZXZpY2UpCiAgICAgICAgeCA9IHNlbGYudG9rKHRva2VucykgKyBzZWxmLnBvcyhwb3NpdGlvbnMpW05vbmUsIDosIDpdCiAgICAgICAgZm9yIGJsb2NrIGluIHNlbGYuYmxvY2tzOgogICAgICAgICAgICB4ID0gYmxvY2soeCkKICAgICAgICBsb2dpdHMgPSBzZWxmLmhlYWQoc2VsZi5maW5hbF9sbih4KSkKICAgICAgICBsb3NzID0gTm9uZQogICAgICAgIGlmIHRhcmdldHMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGxvc3MgPSBGLmNyb3NzX2VudHJvcHkobG9naXRzLnJlc2hhcGUoLTEsIGxvZ2l0cy5zaGFwZVstMV0pLCB0YXJnZXRzLnJlc2hhcGUoLTEpKQogICAgICAgIHJldHVybiBsb2dpdHMsIGxvc3MK')
assert hashlib.sha256(raw).hexdigest()=='e0fcac285918b66b13c8f694e6d625a0691c9b59221b43043e63f46b25ddcc07', 'EMBEDDED_SOURCE_SHA_MISMATCH:p02_source_exact.py'
p=ROOT/'p02_source_exact.py'
if p.exists() and p.read_bytes()!=raw: raise RuntimeError('SOURCE_CHANGED:p02_source_exact.py')
p.write_bytes(raw)
raw=base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJUVE0gVDMgMjAgY29udGludW91cyB2cyAxMCBSRUFEWS1sYXN0IGNoZWNrcG9pbnQgcmVzdG9yZSArMTAgc3ludGhldGljIFFBLgoKUnVuIG9uIENvbGFiIHdpdGggLS1kcml2ZS1yb290IHBvaW50aW5nIGF0IG1vdW50ZWQgL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9UVE0vLi4uCk5vIFRpbnlTdG9yaWVzIG9yIHNjaWVuY2UuIEluZGVwZW5kZW50IGV2aWRlbmNlIGluIHN0cmljdCBuZXcgcnVuX2lkLiBUMyB1c2VzIGV4YWN0IFAwMi4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKaW1wb3J0IGFyZ3BhcnNlLCBoYXNobGliLCBpbXBvcnRsaWIudXRpbCwganNvbiwgbWF0aCwgb3MsIHJhbmRvbSwgc3lzLCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHRvcmNoCgpTRUVEPTIwMjYxMTA4ClAwMl9CTE9CPSc5NzYyMzQzZTA2Y2Y4MDBiNTA1NzRjMjg3MWFlMWVkNTg1MzdmYjE4JwpDT05GSUc9ZGljdCh2b2NhYl9zaXplPTIwNDgsbWF4X3NlcV9sZW49MjU2LG5fZW1iZD0yNTYsbl9sYXllcj00LG5faGVhZD00LG5vcm1fdHlwZT0nbGF5ZXJub3JtJykKVE9UQUw9MjA7IEZJUlNUPTEwCgoKZGVmIGZpbGVfc2hhKHApOgogICAgaD1oYXNobGliLnNoYTI1NigpCiAgICB3aXRoIG9wZW4ocCwncmInKSBhcyBmOgogICAgICAgIGZvciBiIGluIGl0ZXIobGFtYmRhOmYucmVhZCgxPDwyMCksYicnKTpoLnVwZGF0ZShiKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgYXRvbWljX2pzb24ocGF0aCxkYXRhKToKICAgIHRlbXA9cGF0aC53aXRoX3N1ZmZpeChwYXRoLnN1ZmZpeCsnLnBhcnQnKQogICAgaWYgdGVtcC5leGlzdHMoKSBvciBwYXRoLmV4aXN0cygpOnJhaXNlIEZpbGVFeGlzdHNFcnJvcignTk9fT1ZFUldSSVRFX1JFQURZOicrc3RyKHBhdGgpKQogICAgd2l0aCBvcGVuKHRlbXAsJ3gnLGVuY29kaW5nPSd1dGY4JykgYXMgZjoKICAgICAgICBqc29uLmR1bXAoZGF0YSxmLHNvcnRfa2V5cz1UcnVlLGluZGVudD0yKTtmLndyaXRlKCdcbicpO2YuZmx1c2goKTtvcy5mc3luYyhmLmZpbGVubygpKQogICAgb3MucmVwbGFjZSh0ZW1wLHBhdGgpCgoKZGVmIGxvYWRfcDAyKHBhdGgpOgogICAgYj1wYXRoLnJlYWRfYnl0ZXMoKQogICAgaWRlbnQ9aGFzaGxpYi5zaGExKGInYmxvYiAnK3N0cihsZW4oYikpLmVuY29kZSgpK2InXDAnK2IpLmhleGRpZ2VzdCgpCiAgICBpZiBpZGVudCE9UDAyX0JMT0I6cmFpc2UgUnVudGltZUVycm9yKCdQMDJfU09VUkNFX1BJTl9NSVNNQVRDSDonK2lkZW50KQogICAgc3BlYz1pbXBvcnRsaWIudXRpbC5zcGVjX2Zyb21fZmlsZV9sb2NhdGlvbigndHRtX3QzX3AwMl9leGFjdCcscGF0aCkKICAgIG1vZHVsZT1pbXBvcnRsaWIudXRpbC5tb2R1bGVfZnJvbV9zcGVjKHNwZWMpO3N5cy5tb2R1bGVzW3NwZWMubmFtZV09bW9kdWxlO3NwZWMubG9hZGVyLmV4ZWNfbW9kdWxlKG1vZHVsZSkKICAgIHJldHVybiBtb2R1bGUKCgpkZWYgYnVpbGRfbW9kZWwocDAyLGRldmljZSk6CiAgICB0b3JjaC5tYW51YWxfc2VlZChTRUVEKQogICAgaWYgZGV2aWNlLnR5cGU9PSdjdWRhJzp0b3JjaC5jdWRhLm1hbnVhbF9zZWVkX2FsbChTRUVEKQogICAgbW9kZWw9cDAyLkdQVChwMDIuUDAyQ29uZmlnKCoqQ09ORklHKSkudG8oZGV2aWNlKQogICAgaWYgc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkpIT0zNzQ5Mzc2OnJhaXNlIEFzc2VydGlvbkVycm9yKCdQQVJBTVNfQ0hBTkdFRCcpCiAgICByZXR1cm4gbW9kZWwKCgpkZWYgdHJhaW5fY29udGV4dChwMDIsZGV2aWNlKToKICAgIG1vZGVsPWJ1aWxkX21vZGVsKHAwMixkZXZpY2UpCiAgICBvcHQ9dG9yY2gub3B0aW0uQWRhbVcobW9kZWwucGFyYW1ldGVycygpLGxyPTNlLTQsYmV0YXM9KC45LC45NSksd2VpZ2h0X2RlY2F5PS4xKQogICAgc2NoPXRvcmNoLm9wdGltLmxyX3NjaGVkdWxlci5MYW1iZGFMUihvcHQsbHJfbGFtYmRhPWxhbWJkYSBzdGVwOiBtaW4oMS4sKHN0ZXArMSkvNC4pIGlmIHN0ZXA8NCBlbHNlICgwLjErMC45KjAuNSooMSttYXRoLmNvcyhtYXRoLnBpKihzdGVwLTQpLzE2KSkpKQogICAgZz10b3JjaC5HZW5lcmF0b3IoZGV2aWNlPSdjcHUnKS5tYW51YWxfc2VlZChTRUVEKzE3KQogICAgcmV0dXJuIG1vZGVsLG9wdCxzY2gsZwoKCmRlZiBkb19zdGVwcyhtb2RlbCxvcHQsc2NoLGcsc3RhcnQsZW5kLGRldmljZSk6CiAgICBsb3NzZXM9W10KICAgIG1vZGVsLnRyYWluKCkKICAgIGZvciBzdGVwIGluIHJhbmdlKHN0YXJ0LGVuZCk6CiAgICAgICAgIyBTeW50aGV0aWMgdG9rZW5zIG9ubHkuIENQVSBnZW5lcmF0b3Iga2VlcHMgc2FtcGxlciBzZXF1ZW5jZSBhY3Jvc3MgQ1VEQS9DUFUuCiAgICAgICAgcmF3PXRvcmNoLnJhbmRpbnQoMCwyMDQ4LCgyLDMzKSxnZW5lcmF0b3I9ZyxkdHlwZT10b3JjaC5sb25nKQogICAgICAgIHgseT1yYXdbOiw6LTFdLnRvKGRldmljZSkscmF3WzosMTpdLnRvKGRldmljZSkKICAgICAgICBvcHQuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgbG9naXRzLGxvc3M9bW9kZWwoeCx5KQogICAgICAgIGlmIG5vdCB0b3JjaC5pc2Zpbml0ZShsb3NzKS5pdGVtKCk6cmFpc2UgUnVudGltZUVycm9yKCdOT05GSU5JVEVfTE9TUycpCiAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgdG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwxLjApCiAgICAgICAgb3B0LnN0ZXAoKTtzY2guc3RlcCgpCiAgICAgICAgbG9zc2VzLmFwcGVuZChmbG9hdChsb3NzLml0ZW0oKSkpCiAgICByZXR1cm4gbG9zc2VzCgoKZGVmIHBheWxvYWQobW9kZWwsb3B0LHNjaCxnLHN0ZXAsZGV2aWNlKToKICAgIHJldHVybiB7J3NjaGVtYSc6J3R0bS50My5mdWxsLnJlc3VtZS5zeW50aGV0aWMudjEnLCdzdGVwJzpzdGVwLCdzZWVkJzpTRUVELAogICAgICAnY29uZmlnJzpDT05GSUcsJ3AwMl9zb3VyY2VfYmxvYic6UDAyX0JMT0IsCiAgICAgICdtb2RlbCc6bW9kZWwuc3RhdGVfZGljdCgpLCdvcHRpbWl6ZXInOm9wdC5zdGF0ZV9kaWN0KCksCiAgICAgICdzY2hlZHVsZXInOnNjaC5zdGF0ZV9kaWN0KCksJ2JhdGNoX2dlbmVyYXRvcl9zdGF0ZSc6Zy5nZXRfc3RhdGUoKSwKICAgICAgJ3RvcmNoX3JuZ19jcHUnOnRvcmNoLmdldF9ybmdfc3RhdGUoKSwKICAgICAgJ3RvcmNoX3JuZ19jdWRhJzp0b3JjaC5jdWRhLmdldF9ybmdfc3RhdGVfYWxsKCkgaWYgZGV2aWNlLnR5cGU9PSdjdWRhJyBlbHNlIFtdLAogICAgICAncHl0aG9uX3JhbmRvbV9zdGF0ZSc6bGlzdChyYW5kb20uZ2V0c3RhdGUoKSksCiAgICAgICd0b3JjaF92ZXJzaW9uJzpzdHIodG9yY2guX192ZXJzaW9uX18pLCAnZGV2aWNlX3R5cGUnOmRldmljZS50eXBlLAogICAgICAndHJhaW5pbmdfa2luZCc6J1NZTlRIRVRJQ19SRVNVTUVfUUFfT05MWSd9CgoKZGVmIHJlc3RvcmUocGF5bG9hZCxwMDIsZGV2aWNlKToKICAgIGlmIHBheWxvYWRbJ3NjaGVtYSddIT0ndHRtLnQzLmZ1bGwucmVzdW1lLnN5bnRoZXRpYy52MScgb3IgcGF5bG9hZFsnc3RlcCddIT1GSVJTVCBvciBwYXlsb2FkWydjb25maWcnXSE9Q09ORklHIG9yIHBheWxvYWRbJ3AwMl9zb3VyY2VfYmxvYiddIT1QMDJfQkxPQiBvciBwYXlsb2FkWydkZXZpY2VfdHlwZSddIT1kZXZpY2UudHlwZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0NIRUNLUE9JTlRfTUVUQURBVEFfT1JfREVWSUNFX01JU01BVENIJykKICAgIG1vZGVsLG9wdCxzY2gsZz10cmFpbl9jb250ZXh0KHAwMixkZXZpY2UpCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QocGF5bG9hZFsnbW9kZWwnXSxzdHJpY3Q9VHJ1ZSkKICAgIG9wdC5sb2FkX3N0YXRlX2RpY3QocGF5bG9hZFsnb3B0aW1pemVyJ10pCiAgICBzY2gubG9hZF9zdGF0ZV9kaWN0KHBheWxvYWRbJ3NjaGVkdWxlciddKQogICAgZy5zZXRfc3RhdGUocGF5bG9hZFsnYmF0Y2hfZ2VuZXJhdG9yX3N0YXRlJ10pCiAgICByYW5kb20uc2V0c3RhdGUodHVwbGUocGF5bG9hZFsncHl0aG9uX3JhbmRvbV9zdGF0ZSddKSkKICAgIHRvcmNoLnNldF9ybmdfc3RhdGUocGF5bG9hZFsndG9yY2hfcm5nX2NwdSddKQogICAgaWYgZGV2aWNlLnR5cGU9PSdjdWRhJzp0b3JjaC5jdWRhLnNldF9ybmdfc3RhdGVfYWxsKHBheWxvYWRbJ3RvcmNoX3JuZ19jdWRhJ10pCiAgICByZXR1cm4gbW9kZWwsb3B0LHNjaCxnCgoKZGVmIHRkaWZmKGEsYik6CiAgICByZXR1cm4gbWF4KGZsb2F0KChhW2tdLmRldGFjaCgpLmNwdSgpLWJba10uZGV0YWNoKCkuY3B1KCkpLmFicygpLm1heCgpKSBmb3IgayBpbiBhKQoKCmRlZiBvcHRpbWl6ZXJfdGVuc29ycyhvcHQpOgogICAgcmV0dXJuIFt4LmRldGFjaCgpLmNwdSgpLmNsb25lKCkgZm9yIHN0YXRlIGluIG9wdC5zdGF0ZS52YWx1ZXMoKSBmb3IgeCBpbiBzdGF0ZS52YWx1ZXMoKSBpZiB0b3JjaC5pc190ZW5zb3IoeCldCgoKZGVmIG9uZV90cmlhbChwMDIsZGV2aWNlLGRyaXZlX3Jvb3QscnVuX2lkKToKICAgIGlmIG5vdCBydW5faWQuaXNpZGVudGlmaWVyKCk6cmFpc2UgUnVudGltZUVycm9yKCdSVU5fSURfTVVTVF9CRV9TQUZFX0lERU5USUZJRVInKQogICAgaWYgZHJpdmVfcm9vdC5pc19zeW1saW5rKCk6cmFpc2UgUnVudGltZUVycm9yKCdEUklWRV9ST09UX1NZTUxJTksnKQogICAgZHJpdmVfcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIGRpcmVjdG9yeT1kcml2ZV9yb290L3J1bl9pZAogICAgaWYgZGlyZWN0b3J5LmV4aXN0cygpOnJhaXNlIEZpbGVFeGlzdHNFcnJvcignUlVOX0lEX0VYSVNUU19OT19SRVRSWTonK3N0cihkaXJlY3RvcnkpKQogICAgZGlyZWN0b3J5Lm1rZGlyKG1vZGU9MG83MDApCiAgICB0b3JjaC5zZXRfbnVtX3RocmVhZHMoMikKICAgIHRvcmNoLnVzZV9kZXRlcm1pbmlzdGljX2FsZ29yaXRobXMoVHJ1ZSkKICAgICMgUm91bmQgQTogY29udGludW91cwogICAgYmFzZSxibyxicyxiZz10cmFpbl9jb250ZXh0KHAwMixkZXZpY2UpCiAgICBhbGxfbG9zc2VzPWRvX3N0ZXBzKGJhc2UsYm8sYnMsYmcsMCxUT1RBTCxkZXZpY2UpCiAgICAjIFJvdW5kIEI6IGludGVycnVwdGVkL2NoZWNrcG9pbnQvcmVzdW1lZC4gU2FtZSBzeW50aGV0aWMgc2VlZCAmIGNvZGUuCiAgICBwYXJ0aWFsLHBvLHBzLHBnPXRyYWluX2NvbnRleHQocDAyLGRldmljZSkKICAgIGJlZm9yZT1kb19zdGVwcyhwYXJ0aWFsLHBvLHBzLHBnLDAsRklSU1QsZGV2aWNlKQogICAgY3A9ZGlyZWN0b3J5LydjaGVja3BvaW50X3N0ZXAxMC5wdCcKICAgIHRtcD1kaXJlY3RvcnkvJ2NoZWNrcG9pbnRfc3RlcDEwLnB0LnBhcnQnCiAgICBpZiBjcC5leGlzdHMoKSBvciB0bXAuZXhpc3RzKCk6cmFpc2UgRmlsZUV4aXN0c0Vycm9yKCdDSEVDS1BPSU5UX0VYSVNUU19OT19PVkVSV1JJVEUnKQogICAgd2l0aCB0bXAub3BlbigneGInKSBhcyBmOgogICAgICAgIHRvcmNoLnNhdmUocGF5bG9hZChwYXJ0aWFsLHBvLHBzLHBnLEZJUlNULGRldmljZSksZikKICAgICAgICBmLmZsdXNoKCk7b3MuZnN5bmMoZi5maWxlbm8oKSkKICAgIG9zLnJlcGxhY2UodG1wLGNwKQogICAgY2hlY2tzdW09ZmlsZV9zaGEoY3ApCiAgICAjIFJFQURZIHdyaXR0ZW4gb25seSBBRlRFUiBmaWxlIGJ5dGVzIG9uIHRoZSBtb3VudGVkIERyaXZlIHBhdGggYXJlIGZzeW5jZWQuCiAgICByZWFkeT17J3NjaGVtYSc6J3R0bS50My5yZWFkeS5sYXN0LnYxJywnc2hhMjU2JzpjaGVja3N1bSwnYnl0ZXMnOmNwLnN0YXQoKS5zdF9zaXplLAogICAgICAgICAgICdmaWxlbmFtZSc6Y3AubmFtZSwnc3RlcCc6RklSU1QsJ3AwMl9naXRfYmxvYic6UDAyX0JMT0IsCiAgICAgICAgICAgJ2NoZWNrcG9pbnRfaW5jbHVkZXMnOlsnbW9kZWwnLCdvcHRpbWl6ZXInLCdzY2hlZHVsZXInLCdybmcnLCdzYW1wbGVyJywnY3Vyc29yJ10sCiAgICAgICAgICAgJ3N5bnRoZXRpY19vbmx5JzpUcnVlfQogICAgYXRvbWljX2pzb24oZGlyZWN0b3J5LydSRUFEWS5qc29uJyxyZWFkeSkKICAgICMgUmVhZCBiYWNrIGZyb20gZHVyYWJsZSBEcml2ZSBwYXRoLCBub3QgZnJvbSBpbi1tZW1vcnkgcGF5bG9hZC4KICAgIG9ic2VydmVkPWpzb24ubG9hZHMoKGRpcmVjdG9yeS8nUkVBRFkuanNvbicpLnJlYWRfdGV4dCgndXRmOCcpKQogICAgaWYgb2JzZXJ2ZWQhPXJlYWR5IG9yIGZpbGVfc2hhKGNwKSE9Y2hlY2tzdW06cmFpc2UgUnVudGltZUVycm9yKCdEUklWRV9SRUFEWV9SRUFEQkFDS19GQUlMRUQnKQogICAgIyBPbmx5IHJlYWQgdHJ1c3RlZCBvd24gbG9jYWwgY2hlY2twb2ludCBhZnRlciBleGFjdCBTSEEuIHdlaWdodHNfb25seT1UcnVlIGVuZm9yY2VzIHNhZmUgdHlwZXMuCiAgICBja3B0PXRvcmNoLmxvYWQoY3AsbWFwX2xvY2F0aW9uPWRldmljZSx3ZWlnaHRzX29ubHk9VHJ1ZSkKICAgIHJlc3VtZWQscm8scnMscmc9cmVzdG9yZShja3B0LHAwMixkZXZpY2UpCiAgICBhZnRlcj1kb19zdGVwcyhyZXN1bWVkLHJvLHJzLHJnLEZJUlNULFRPVEFMLGRldmljZSkKICAgIGxvc3NfZGlmZj1tYXgoYWJzKGEtYikgZm9yIGEsYiBpbiB6aXAoYWxsX2xvc3NlcyxiZWZvcmUrYWZ0ZXIpKQogICAgd2VpZ2h0c19kaWZmPXRkaWZmKGJhc2Uuc3RhdGVfZGljdCgpLHJlc3VtZWQuc3RhdGVfZGljdCgpKQogICAgb3B0aW1BPW9wdGltaXplcl90ZW5zb3JzKGJvKTtvcHRpbUI9b3B0aW1pemVyX3RlbnNvcnMocm8pCiAgICBpZiBsZW4ob3B0aW1BKSE9bGVuKG9wdGltQik6cmFpc2UgUnVudGltZUVycm9yKCdPUFRfU1RBVEVfTEVOR1RIX01JU01BVENIJykKICAgIG9wdGltX2RpZmY9bWF4KGZsb2F0KChhLWIpLmFicygpLm1heCgpKSBmb3IgYSxiIGluIHppcChvcHRpbUEsb3B0aW1CKSkKICAgIHNjaGVkdWxlcl9vaz1icy5zdGF0ZV9kaWN0KCk9PXJzLnN0YXRlX2RpY3QoKQogICAgc2FtcGxlcl9vaz10b3JjaC5lcXVhbChiZy5nZXRfc3RhdGUoKSxyZy5nZXRfc3RhdGUoKSkKICAgIHB5dGhvbl9ybmdfcmVzdG9yZWQ9cmFuZG9tLmdldHN0YXRlKCk9PXR1cGxlKGNrcHRbJ3B5dGhvbl9yYW5kb21fc3RhdGUnXSkKICAgIG9rPWxvc3NfZGlmZj09MCBhbmQgd2VpZ2h0c19kaWZmPT0wIGFuZCBvcHRpbV9kaWZmPT0wIGFuZCBzY2hlZHVsZXJfb2sgYW5kIHNhbXBsZXJfb2sgYW5kIHB5dGhvbl9ybmdfcmVzdG9yZWQKICAgIHJlY2VpcHQ9eydzY2hlbWEnOid0dG0udDMucmVzdW1lLnBhcml0eS5yZXBvcnQudjEnLCdtb2RlJzonU1lOVEhFVElDX09OTFlfTk9fU0NJRU5DRScsCiAgICAgICAgJ2RldmljZSc6c3RyKGRldmljZSksJ3RvcmNoJzpzdHIodG9yY2guX192ZXJzaW9uX18pLCdwMDJfZ2l0X2Jsb2InOlAwMl9CTE9CLAogICAgICAgICdzdGVwc19jb250aWd1b3VzJzpUT1RBTCwnc3RlcF9jaGVja3BvaW50JzpGSVJTVCwKICAgICAgICAnY2hlY2twb2ludF9zaGEyNTYnOmNoZWNrc3VtLCdjaGVja3BvaW50X2J5dGVzJzpjcC5zdGF0KCkuc3Rfc2l6ZSwKICAgICAgICAnY2hlY2twb2ludF9yZWFkeV9sYXN0JzpUcnVlLCdyZWFkeV9yZWFkYmFjayc6VHJ1ZSwKICAgICAgICAnbWF4X2Fic19sb3NzX2RpZmYnOmxvc3NfZGlmZiwnbWF4X2Fic193ZWlnaHRfZGlmZic6d2VpZ2h0c19kaWZmLAogICAgICAgICdtYXhfYWJzX29wdGltaXplcl9kaWZmJzpvcHRpbV9kaWZmLCdzY2hlZHVsZXJfZXhhY3QnOnNjaGVkdWxlcl9vaywKICAgICAgICAnc2FtcGxlcl9zdGF0ZV9leGFjdCc6c2FtcGxlcl9vaywncHl0aG9uX3JuZ19zdGF0ZV9yZXN0b3JlZCc6cHl0aG9uX3JuZ19yZXN0b3JlZCwncGFzcyc6Ym9vbChvayksJ2dwdV9zY2llbmNlX3VzZWQnOkZhbHNlLAogICAgICAgICdzdGF0dXMnOidQQVNTX1NZTlRIRVRJQ19SRVNVTUUnIGlmIG9rIGVsc2UgJ0ZBSUxfU1lOVEhFVElDX1JFU1VNRSd9CiAgICBhdG9taWNfanNvbihkaXJlY3RvcnkvJ1QzX1JFU1VNRV9SRVBPUlQuanNvbicscmVjZWlwdCkKICAgIHJldHVybiByZWNlaXB0CgoKZGVmIG1haW4oKToKICAgIGFwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1wMDInLHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tZHJpdmUtcm9vdCcscmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1ydW4taWQnLHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tZGV2aWNlJyxjaG9pY2VzPSgnY3B1JywnY3VkYScpLGRlZmF1bHQ9J2NwdScpCiAgICBhcmdzPWFwLnBhcnNlX2FyZ3MoKQogICAgZGV2aWNlPXRvcmNoLmRldmljZShhcmdzLmRldmljZSkKICAgIGlmIGRldmljZS50eXBlPT0nY3VkYScgYW5kIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOnJhaXNlIFJ1bnRpbWVFcnJvcignQ09MQUJfQ1VEQV9OT1RfQVZBSUxBQkxFJykKICAgIHJhbmRvbS5zZWVkKFNFRUQpO3RvcmNoLm1hbnVhbF9zZWVkKFNFRUQpCiAgICBpZiBkZXZpY2UudHlwZT09J2N1ZGEnOgogICAgICAgIGlmIGhhc2F0dHIodG9yY2guYmFja2VuZHMuY3VkYS5tYXRtdWwsJ2FsbG93X3RmMzInKTp0b3JjaC5iYWNrZW5kcy5jdWRhLm1hdG11bC5hbGxvd190ZjMyPUZhbHNlCiAgICAgICAgaWYgaGFzYXR0cih0b3JjaC5iYWNrZW5kcy5jdWRubiwnYWxsb3dfdGYzMicpOnRvcmNoLmJhY2tlbmRzLmN1ZG5uLmFsbG93X3RmMzI9RmFsc2UKICAgICMgRXhhY3QgZGV0ZXJtaW5pc3RpYyBHUFUgc2V0dXAgbWF5IHJlcXVpcmUgQ1VCTEFTX1dPUktTUEFDRV9DT05GSUcgYmVmb3JlIGltcG9ydGluZyB0b3JjaCBvbiBDb2xhYi4KICAgIHAwMj1sb2FkX3AwMihQYXRoKGFyZ3MucDAyKSkKICAgIHJlc3VsdD1vbmVfdHJpYWwocDAyLGRldmljZSxQYXRoKGFyZ3MuZHJpdmVfcm9vdCksYXJncy5ydW5faWQpCiAgICBwcmludCgnVFRNX1QzX1JFU1VNRV9SRVNVTFQ9Jytqc29uLmR1bXBzKHJlc3VsdCxzb3J0X2tleXM9VHJ1ZSkpCiAgICBpZiBub3QgcmVzdWx0WydwYXNzJ106c3lzLmV4aXQoMikKCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOm1haW4oKQo=')
assert hashlib.sha256(raw).hexdigest()=='a95c50f20771fbc3137b8be187ae8f220957345d9d3e3eb3a10989fcd4ff8f2b', 'EMBEDDED_SOURCE_SHA_MISMATCH:t3_resume_smoke.py'
p=ROOT/'t3_resume_smoke.py'
if p.exists() and p.read_bytes()!=raw: raise RuntimeError('SOURCE_CHANGED:t3_resume_smoke.py')
p.write_bytes(raw)
print('TTM_T3_NOTEBOOK_SOURCE_READBACK=PASS')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive/TTM/ttm-small-pilot')
assert Path('/content/drive/MyDrive').is_dir(), 'GOOGLE_DRIVE_MOUNT_NOT_READY'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID='t3_resume_synthetic_20261108_v1'
assert not (DRIVE_ROOT/RUN_ID).exists(), 'T3_RUN_ID_CONSUMED_NO_RERUN'
print('TTM_T3_COLAB_DRIVE_MOUNT=PASS')


In [ ]:
import torch, sys, subprocess, json, os, hashlib, platform
if not torch.cuda.is_available():
    raise RuntimeError('COLAB_CUDA_GPU_NOT_ALLOCATED; T3 Colab GPU test cannot silently use CPU')
print('TTM_T3_GPU='+torch.cuda.get_device_name(0))
print('TTM_T3_TORCH='+str(torch.__version__))
run=subprocess.run([sys.executable,str(ROOT/'t3_resume_smoke.py'),
 '--p02',str(ROOT/'p02_source_exact.py'), '--drive-root',str(DRIVE_ROOT),
 '--run-id',RUN_ID,'--device','cuda'],capture_output=True,text=True,timeout=600)
print(run.stdout)
if run.stderr: print(run.stderr[-4000:])
if run.returncode:raise RuntimeError('TTM_T3_COLAB_RESUME_FAIL:'+str(run.returncode))
report_path=DRIVE_ROOT/RUN_ID/'T3_RESUME_REPORT.json'
report=json.loads(report_path.read_text('utf8'))
assert report['pass'] and report['device'].startswith('cuda'), 'BAD_COLAB_T3_REPORT'
freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
freeze_file=DRIVE_ROOT/RUN_ID/'COLAB_PIP_FREEZE.txt'
assert not freeze_file.exists()
freeze_file.write_text(freeze,encoding='utf8')
env={'schema':'ttm.t3.colab.env.v1','python':sys.version,'platform':platform.platform(),
 'torch':str(torch.__version__),'cuda':str(torch.version.cuda),
 'gpu_name':torch.cuda.get_device_name(0),
 'pip_freeze_sha256':hashlib.sha256(freeze.encode()).hexdigest(),
 'report_sha256':hashlib.sha256(report_path.read_bytes()).hexdigest(),
 'science_executed':False}
(DRIVE_ROOT/RUN_ID/'COLAB_ENVIRONMENT.json').write_text(json.dumps(env,indent=2,sort_keys=True)+'\n',encoding='utf8')
print('TTM_T3_COLAB_MOUNTED_DRIVE_RESUME=PASS; NEED_FLUSH_AND_EXTERNAL_READBACK')
print('TTM_T3_REPORT_SHA256='+env['report_sha256'])


In [ ]:
from google.colab import drive
# Flush/unmount only when all preceding cells PASS. This is not a network-independent
# durability guarantee: check files through Google Drive after it finishes.
drive.flush_and_unmount()
print('TTM_T3_COLAB_DRIVE_FLUSH=COMPLETED; INDEPENDENT_DRIVE_READBACK_STILL_REQUIRED')
